# Demo **Paraphrasing**

Our imposter approach needs models to reword an original text keeping the same meaning.
Ideally, we want to model the way texts originated.
This is dependent on the text's genre, i.e. a student essay originates from a task description and a list of URLs while a new article originates from a list of additional sources or from spectating an event.
We implemented both a naive approach to rewording a text and a more sophisticated one:
- given a prompt and a text, the naive approach uses a language model to paraphrase the text
- given a prompt and a text LLM A is used to extract the text's genre, tone and summary of bullet points, then LLM B is used to reword the text in the same genre and tone based on the summary of bullet points

In [ ]:
from pathlib import Path
import os
import textwrap
import sys

# Add the root of the project to Python path
sys.path.append(os.path.abspath(".."))
from genai_detection.config import CONFIG
from genai_detection.paraphrasing.paraphraser import (
    T5ChatGPTParaphraser,
    T5GooglePAWSParaphraser,
    BlabladorParaphraser,
    TranslationParaphraser,
    BulletPointParaphraser,
    OllamaParaphraser,
    TaskParaphraser,
    TopicParaphraser,
    TitleParaphraser,
    SAIAParaphraser,
)

## Original text
We use a CNN news article from 23.06.2025/ 04.07.2025 as an example.

In [ ]:
path2datasets = (
    Path(os.getcwd()).resolve().parent / "data" / "datasets" / "custom_texts"
)
data_category = "News"  # This is used for plot titles in the notebook
path2results = Path(os.getcwd()).resolve().parent / CONFIG.SAVE_PATH / "paraphrasing"
assert path2datasets.exists(), f"Path to datasets {path2datasets} does not exist."
# file_name = "cnn_230625"    # USA attacks Iran
file_name = "cnn_040725"  # Dalai Lama
original_text = open(path2datasets / f"{file_name}.txt").read()

print(textwrap.fill(original_text, width=80))

  For much of the past century, the Dalai Lama has been the living embodiment of
Tibet’s struggle for greater freedoms under Chinese Communist Party rule,
sustaining the cause from exile even as an increasingly powerful Beijing has
become ever more assertive in suppressing it.  As his 90th birthday approaches
this Sunday, the spiritual leader for millions of followers of Tibetan Buddhism
worldwide is bracing for a final showdown with Beijing: the battle over who will
control his reincarnation.  On Wednesday, the Dalai Lama announced that he will
have a successor after his death, and that his office will have the sole
authority to identify his reincarnation.  “I am affirming that the institution
of the Dalai Lama will continue,” the Nobel Peace laureate said in a video
message to religious elders gathering in Dharamshala, India, where he has found
refuge since Chinese communist troops put down an armed uprising in his
mountainous homeland in 1959.  The cycle of rebirth lies at the core 

## Naive Approach

In [ ]:
paraphrasers = {
    "T5_ChatGPT": T5ChatGPTParaphraser(),
    "T5_Google_PAWS": T5GooglePAWSParaphraser(),
    # "Blablador": BlabladorParaphraser(
    #     model_id="1 - Llama3 405 the best general model and big context size"
    # ),
    "Ollama": OllamaParaphraser(model_id=CONFIG.OLLAMA_VERSION),
    "SAIA": SAIAParaphraser(model_id=CONFIG.SAIAI_VERSION),
}
if "Blablador" in paraphrasers.keys():
    print(
        "Available Blablador models:\n",
        paraphrasers["Blablador"]._get_available_models(verbose=False),
    )

Using T5ChatGPTParaphraser
Loaded T5ChatGPTParaphraser tokenizer
Loaded T5ChatGPTParaphraser model
Using T5GooglePAWSParaphraser


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Loaded T5GooglePAWSParaphraser tokenizer
Loaded T5GooglePAWSParaphraser model


In [ ]:
n_responses = 1  # number of paraphrases to generate
max_length = 512  # Maximum length of the generated paraphrase
temperature = 0.7  # Controls the randomness of the output. Lower values make the output more deterministic.

prompts = [
    "Paraphrase the text above and output only the paraphrased version.",
    "For the text above: First, extract bullet points capturing the main ideas, then create a text based on these bullet points. Only output the final text (i.e. do not output the bullet points or any additional chain of thoughts).",
    "For the text above: Paraphrase the sentence by first identifying the main subject, verb, and object. Then find synonyms for each and construct a new sentence. Only output the final paraphrased sentence.",
    "For the text above: Paraphrase the sentence using the same tone as the original with approximately the same number of words.",
    "For the text above: Paraphrase this sentence. Do not change the meaning, but use different words and structure. Output only the paraphrased sentence.",
]

In [ ]:
for paraphraser_name, paraphraser in paraphrasers.items():
    print(f"\n{'='*40}")
    print(f"\nUsing paraphraser: {paraphraser_name}")
    for prompt in prompts:
        print(f"\nPrompt: {prompt}")
        paraphrases = paraphraser.paraphrase(
            original_text,
            n_responses=n_responses,
            max_length=max_length,
            temperature=temperature,
            prompt=prompt,
        )
        for i, paraphrase in enumerate(paraphrases):
            print(f"Paraphrase {i+1}: {textwrap.fill(paraphrase, width=80)}")



Using paraphraser: T5_ChatGPT

Prompt: Paraphrase the text above and output only the paraphrased version.
T5 model


Group Beam Search is scheduled to be moved to a `custom_generate` repository in v4.55.0. To prevent loss of backward compatibility, add `trust_remote_code=True` to your `generate` call.


Paraphrase 1: Tibetan Buddhists are bracing themselves for a potential confrontation with
Beijing, who has been resisting the Chinese Communist Party's efforts to bring
about their own political and economic reforms. Tibet's current Dalai Lama is
viewed as a complex figure who represents the ideal conditions for Tibetan
unity, including freedom of religion and self-determination.

Prompt: For the text above: First, extract bullet points capturing the main ideas, then create a text based on these bullet points. Only output the final text (i.e. do not output the bullet points or any additional chain of thoughts).
T5 model
Paraphrase 1: Tibetan Buddhists are bracing themselves for a potential confrontation with
Beijing, who has been actively resisting the Chinese Communist Party's rule for
decades. As his 90th birthday approaches on Sunday, the Dalai Lama announced
that he will have an exclusive opportunity to identify his successor in person
and not as ordinary figures. The Dalai Lama's 

KeyboardInterrupt: 

## Bullet Point Approach

Ollama models are by far the best ones for extracting the genre, tone and summary of bullet points and returning them in a structured way.
Hence, we use the Ollama model as the text extractor and any other model as the text generator.

In [ ]:
models = {
    "text_extractor": {
        "name": "Ollama-latest",
        "model": SAIAParaphraser(model_id=CONFIG.SAIAI_VERSION),
    },
    "text_generator": {
        "name": "Ollama-latest",
        "model": SAIAParaphraser(model_id=CONFIG.SAIAI_VERSION),
    },
}
bullet_point_paraphraser = BulletPointParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = bullet_point_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")
# save to file
# with open(path2datasets / f"paraphrase_{file_name}_e={models["text_extractor"]['name']}_g={models["text_generator"]['name']}_{i+1}.txt", "w") as f:
#     f.write(paraphrase)

Paraphrase 1:
 Title: A Contested Legacy: The Dalai Lama's Imminent Reincarnation and the Struggle for Tibetan Autonomy

In a significant move that underscores the complex interplay of spiritual leadership and political struggle, His Holiness the 14th Dalai Lama is preparing for the selection process of his successor. The impending event carries profound implications for Tibetan Buddhism and the ongoing efforts to secure greater freedoms under Chinese Communist Party (CCP) rule in Tibet.

The reincarnation of the Dalai Lama, a revered spiritual figure, is not just a matter of religious significance; it is also an emotive symbol of Tibet's cultural identity and political aspirations. The role of the Dalai Lama has been instrumental in preserving Tibetan Buddhism and its unique culture, making the issue of his reincarnation a contentious one.

Recently, the Dalai Lama's office announced that it will retain sole authority to identify his successor. This declaration contradicts China's ass

## Task Paraphraser
The task paraphraser extracts the task, tone and genre from the text and then generates a paraphrase based on this information.

In [ ]:
task_paraphraser = TaskParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = task_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

Paraphrase 1:
 Title: The Enduring Controversy over the Dalai Lama's Reincarnation: Political Implications for Tibet and the Himalayas

In the towering peaks of the Himalayan range, a spiritual and political conundrum unfolds, one that has reverberated across centuries. At its heart lies the enigmatic figure of His Holiness, the 14th Dalai Lama, and the contentious issue of his reincarnation, intertwined with the complex relationship between Tibet and China.

The Dalai Lama, revered by many as a living god, is not just a spiritual leader for Tibetans but also a symbol of cultural identity and political resistance. His reincarnation is seen as a sacred continuity that ensures the preservation of Buddhist teachings and Tibetan culture. The current Dalai Lama has stated, "The reincarnation system embodies the unique cultural identity of the Tibetan people."

However, this tradition is perceived as a threat by the Chinese Communist Party (CCP), which views it as a challenge to its authorit

## Topic Paraphraser
The topic paraphraser extracts the topic, tone and genre from the text and then generates a paraphrase based on this information.

In [ ]:
topic_paraphraser = TopicParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = topic_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

Paraphrase 1:
 Title: The Tibetan Struggle for Freedom: An Ongoing Quest Under Chinese Rule

In the heart of the Himalayas, the autonomous region of Tibet stands as a beacon of cultural richness and spiritual wisdom, reflecting the resilience and determination of its people in their pursuit of freedom. This article delves into the ongoing struggle for self-determination faced by the Tibetans under Chinese rule, with insights from His Holiness, the 14th Dalai Lama.

The Tibetan Plateau, often referred to as the 'Roof of the World,' has been a region steeped in ancient history and spiritual traditions for centuries. However, since China's military occupation in 1950, Tibet has faced an unprecedented period of suppression and cultural erosion. The Tibetan people have endured numerous human rights abuses, including forced assimilation, religious persecution, and environmental degradation under the Chinese regime.

His Holiness, the Dalai Lama, has been a tireless advocate for Tibet's indep

## Title Paraphraser
The title paraphraser extracts the title, tone and genre from the text and then generates a paraphrase based on this information.

In [ ]:
title_paraphraser = TitleParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
)
paraphrases = title_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

Paraphrase 1:
 Title: Dalai Lama's Successor: A Geopolitical Battle Over Tibet's Future

In the vibrant tapestry of global politics, the impending succession of the 14th Dalai Lama has emerged as a pivotal point, igniting a geopolitical struggle over the future of Tibet. The Dalai Lama, who turned 86 this year, has long maintained that he is the last of his lineage and that his successor will be chosen through reincarnation. However, with China asserting its control over Tibetan affairs, the traditional process faces significant challenges, casting a shadow over the spiritual and political future of Tibet.

The Dalai Lama's lineage traces back six centuries, with each incarnation believed to be a manifestation of Avalokitesvara, the embodiment of compassion in Buddhism. The current Dalai Lama was recognized at the tender age of two by senior Tibetan lamas and Chinese officials, following the death of his predecessor in 1935. His leadership has been marked by a steadfast commitment to p

# Translation Paraphraser

In [ ]:
translation_paraphraser = TranslationParaphraser(
    text_extractor=models["text_extractor"]["model"],
    text_generator=models["text_generator"]["model"],
    language="French",
)
paraphrases = translation_paraphraser.paraphrase(
    text=original_text, prompt=None, n_responses=n_responses, max_length=max_length
)
for i, paraphrase in enumerate(paraphrases):
    print(f"Paraphrase {i+1}:\n{paraphrase}\n")

## Batch Processing

In [ ]:
# Batch paraphrasing
texts = [
    "The sky is blue.",
    "I love programming.",
    "Artificial intelligence is fascinating.",
]
paraphrased_batch = bullet_point_paraphraser.paraphrase_batch(texts)
for original, paraphrased in zip(texts, paraphrased_batch):
    print(f"Original: {original} | Paraphrased: {paraphrased}")

Original: The sky is blue. | Paraphrased: [' Clear Blue Sky Today', ' Azure expanse unfolds.', ' Clear blue skies prevail.', ' Clear, Blue Sky Presently', ' Sky is Azure Now']
Original: I love programming. | Paraphrased: ['Savoring the Artistry of Modern Programming', 'Experience the Joy of Modern Programming', ' Delight in Current Programming Pursuits', 'Delightfully Engaging with Current Programming Solutions', 'Experience Thrilling Programming Progress']
Original: Artificial intelligence is fascinating. | Paraphrased: ['Artificial Intelligence: The Development of Intelligent Machines Capable of Learning, Reasoning, Problem-Solving, and Perception.', 'Artificial Intelligence: Development of Intelligent Machines Capable of Learning, Reasoning, Problem-Solving, and Perception', 'Artificial Intelligence: Machine Learning, Reasoning, Problem Solving, Perception', 'AI: Intelligent Machines for Learning, Reasoning, Problem-Solving, Perception', 'Artificial Intelligence: Constructing Intell